# 04 · Assignment: model comparison report

1. Write **5 test questions** on one topic you know well
2. Run each on **2 models** (for example Gemini and Groq) at temperature 0.2
3. Save the answer, tokens in and out, and latency to `results.csv`
4. Rate each answer in the CSV: **correct**, **partly correct** or **wrong**
5. In your README: which model would you choose, and why?

**Submit** your GitHub repo link on Google Classroom.

These 5 questions are your first *test set*. You will reuse them later in the course.
Do not put personal data in your questions.

In [1]:
import os
import time
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # reads the .env file in this folder

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.8-flash")
GROQ_MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-20b")

# Never print the key itself, only whether it was found
print("Gemini key found:", bool(os.getenv("GEMINI_API_KEY")))
print("Groq key found:  ", bool(os.getenv("GROQ_API_KEY")))

Gemini key found: True
Groq key found:   True


## Step 1 · Your test questions

In [2]:
# TODO: replace these with 5 questions on a topic you know well
QUESTIONS = [
    "What is a list in Python?",
    "What is the difference between a list and a tuple?",
    "When should I use a dictionary instead of a list?",
    "What does list(range(3, 10, 2)) return?",
    "Why does a = [1,2]; b = a; b.append(3) change a too?",
]

## Step 2 · The models to compare

In [3]:
MODELS = {
    "gemini": (OpenAI(api_key=os.getenv("GEMINI_API_KEY"), base_url=GEMINI_BASE_URL), GEMINI_MODEL),
    "groq": (OpenAI(api_key=os.getenv("GROQ_API_KEY"), base_url=GROQ_BASE_URL), GROQ_MODEL),
}

## Step 3 · Run every question on every model

In [4]:
results = []

for provider, (client, model) in MODELS.items():
    for q in QUESTIONS:
        for attempt in range(1, 4):
            start = time.perf_counter()
            try:
                r = client.chat.completions.create(
                    model=model,
                    messages=[{"role": "user", "content": q}],
                    temperature=0.2,
                    max_tokens=2000,
                    timeout=120,
                )
                latency = time.perf_counter() - start
                results.append({
                    "provider": provider,
                    "model": model,
                    "question": q,
                    "answer": r.choices[0].message.content.strip(),
                    "tokens_in": r.usage.prompt_tokens if r.usage else None,
                    "tokens_out": r.usage.completion_tokens if r.usage else None,
                    "latency_s": round(latency, 2),
                    "rating": "",
                })
                print(f"{provider}: done ({latency:.2f}s)")
                break
            except Exception as e:
                wait = 20 * attempt
                print(f"{provider} attempt {attempt} failed: {type(e).__name__}. Waiting {wait}s")
                time.sleep(wait)
        else:
            print(f"{provider} SKIPPED: {q}")

gemini attempt 1 failed: APITimeoutError. Waiting 20s
gemini attempt 2 failed: APITimeoutError. Waiting 40s
gemini: done (85.56s)
gemini attempt 1 failed: InternalServerError. Waiting 20s
gemini: done (28.41s)
gemini: done (353.17s)
gemini attempt 1 failed: InternalServerError. Waiting 20s
gemini: done (232.20s)
gemini: done (73.24s)
groq: done (3.87s)
groq: done (2.32s)
groq: done (3.06s)
groq: done (1.12s)
groq: done (2.54s)


## Step 4 · Save to results.csv

In [5]:
import csv

with open("results.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(results[0].keys()))
    writer.writeheader()
    writer.writerows(results)

print("Saved", len(results), "rows to results.csv")

Saved 10 rows to results.csv


## Step 5 · Rate the answers and write your README

Open `results.csv`, fill in the **rating** column for each row, then answer in your README:

- Which model gave more correct answers?
- Which was faster on average?
- Which would you choose for this topic, and why?

Optional: add error handling and retries from notebook 02.